# Classifying Autism vs. Control from Brain Connectivity

In [13]:
import numpy as np
import pandas as pd
from nilearn import datasets

from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

In [21]:
abide = datasets.fetch_abide_pcp(
    n_subjects=50,
    pipeline="cpac",
    derivatives=["rois_cc200"],
)

roi_time_series = abide.rois_cc200          # list of arrays: (timepoints, ~200 regions)
phenotypic = pd.DataFrame(abide.phenotypic)  # subject info, incl. diagnosis
print(phenotypic)
# DX_GROUP: 1 = Autism, 2 = Control
print(f"Number of subjects downloaded: {len(roi_time_series)}")
print(f"Shape of first subject's data: {roi_time_series[0].shape}")
print(phenotypic["DX_GROUP"].value_counts())

[fetch_abide_pcp] Dataset directory found: C:\Users\cchhe\nilearn_data\ABIDE_pcp

     i  Unnamed: 0  SUB_ID   X  subject SITE_ID       FILE_ID  DX_GROUP  \
1    1           2   50003   2    50003    PITT  Pitt_0050003         1   
2    2           3   50004   3    50004    PITT  Pitt_0050004         1   
3    3           4   50005   4    50005    PITT  Pitt_0050005         1   
4    4           5   50006   5    50006    PITT  Pitt_0050006         1   
5    5           6   50007   6    50007    PITT  Pitt_0050007         1   
6    6           7   50008   7    50008    PITT  Pitt_0050008         1   
8    8           9   50010   9    50010    PITT  Pitt_0050010         1   
9    9          10   50011  10    50011    PITT  Pitt_0050011         1   
10  10          11   50012  11    50012    PITT  Pitt_0050012         1   
11  11          12   50013  12    50013    PITT  Pitt_0050013         1   
12  12          13   50014  13    50014    PITT  Pitt_0050014         1   
13  13          14   50015  14    50015    PITT  Pitt_0050015         1   
14  14          15   5001

In [15]:
def connectivity_features(ts):
    """
    Convert a (timepoints x regions) time series into a flattened
    upper-triangle correlation vector -- this becomes one subject's
    feature row.
    """
    corr_matrix = np.corrcoef(ts.T)  # transpose: rows=regions, cols=timepoints
    # Get indices of the upper triangle, excluding the diagonal (which is
    # always 1.0 -- a region is always perfectly correlated with itself)
    upper_indices = np.triu_indices_from(corr_matrix, k=1)
    return corr_matrix[upper_indices]

In [16]:
feature_rows = []
valid_indices = []

for i, ts in enumerate(roi_time_series):
    try:
        features = connectivity_features(ts)
        feature_rows.append(features)
        valid_indices.append(i)
    except Exception as e:
        print(f"Subject {i} skipped: {e}")

X = np.array(feature_rows)                       # shape: (n_subjects, n_connections)
y = phenotypic.iloc[valid_indices]["DX_GROUP"].values  # 1 = ASD, 2 = Control

print(f"Feature matrix shape: {X.shape}")
print(f"Labels shape: {y.shape}")


nan_mask = np.isnan(X).any(axis=1)
print(f"Subjects with NaN values: {nan_mask.sum()} out of {len(X)}")

# Drop those subjects entirely
X = X[~nan_mask]
y = y[~nan_mask]

print(f"Feature matrix shape after dropping NaNs: {X.shape}")
print(f"Labels shape after dropping NaNs: {y.shape}")

Feature matrix shape: (50, 19900)
Labels shape: (50,)
Subjects with NaN values: 1 out of 50
Feature matrix shape after dropping NaNs: (49, 19900)
Labels shape after dropping NaNs: (49,)


c:\Users\cchhe\anaconda3\Lib\site-packages\numpy\lib\_function_base_impl.py:2999: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\cchhe\anaconda3\Lib\site-packages\numpy\lib\_function_base_impl.py:3000: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [17]:
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, C=0.1)  # C=0.1 = fairly strong regularization
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")

print(f"Cross-validated accuracy per fold: {scores}")
print(f"Mean accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")

# Compare to a "dumb" baseline: always predicting the majority class
majority_class_fraction = max(np.mean(y == 1), np.mean(y == 2))
print(f"Baseline (always guess majority class): {majority_class_fraction:.3f}")

Cross-validated accuracy per fold: [0.7        0.7        0.8        0.7        0.66666667]
Mean accuracy: 0.713 (+/- 0.045)
Baseline (always guess majority class): 0.510


In [18]:
model.fit(X, y)
logreg = model.named_steps["logisticregression"]
coefficients = logreg.coef_[0]

top_indices = np.argsort(np.abs(coefficients))[::-1][:10]
print("Top 10 most influential region-pair connections:")
for idx in top_indices:
    print(f"  connection #{idx}: weight = {coefficients[idx]:.4f}")

Top 10 most influential region-pair connections:
  connection #17465: weight = 0.0088
  connection #7477: weight = 0.0084
  connection #17544: weight = 0.0084
  connection #11570: weight = -0.0083
  connection #3030: weight = 0.0081
  connection #19471: weight = -0.0079
  connection #17520: weight = 0.0078
  connection #10525: weight = 0.0076
  connection #3032: weight = 0.0075
  connection #6194: weight = -0.0074
